# P4-A4 — Reflection (agents that critique and improve their own work)

The final pattern: **self-correction**. Instead of accepting the model's first draft, you loop — *generate → critique → revise* — until the output is good enough (or you hit a cap). One model wears two hats: a **generator** and a **critic**.

This is the same loop shape you keep building (a graph that cycles until a stop condition), but here the cycle *improves quality* rather than calling tools. It's how you squeeze better output out of the same model.

Graph: `generate → reflect → (good? END : back to generate)`.

In [1]:
# --- Setup (given) ---
from typing_extensions import TypedDict
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

load_dotenv()
llm = ChatOpenAI(model='gpt-4o-mini')
MAX_ITERS = 3

def chat(prompt):
    """One-shot LLM call, returns text."""
    return llm.invoke(prompt).content

# Shared state that flows through the loop
class ReflectState(TypedDict):
    task: str         # what we're trying to produce
    draft: str        # the current attempt
    critique: str     # the latest critique (empty on first pass)
    iterations: int   # how many drafts so far

print('ready — MAX_ITERS =', MAX_ITERS)

ready — MAX_ITERS = 3


## Task 1 — The `generate` and `reflect` nodes
Write two node functions:
- **`generate(state)`** → produce a `draft`. On the first pass (no critique) just attempt the `task`; on later passes, *revise the previous draft using the critique*. Increment `iterations`. Print the draft.
- **`reflect(state)`** → critique the current `draft` against the `task`. If it's genuinely good, start the reply with `APPROVED`; otherwise give specific, actionable feedback. Store it in `critique`. Print it.
<br>Hint: each returns a dict updating only the keys it changes, e.g. `return {'draft': ..., 'iterations': state['iterations'] + 1}`.

In [2]:
# TODO: def generate(state): ...   and   def reflect(state): ...

def generate(state):
    task = state['task']
    if not state['critique']:                       # first pass: attempt from scratch
        prompt = f"Task: {task}\n\nWrite your best attempt."
    else:                                           # later passes: revise using the critique
        prompt = (
            f"Task: {task}\n\n"
            f"Your previous draft:\n{state['draft']}\n\n"
            f"Critique to address:\n{state['critique']}\n\n"
            "Rewrite the draft to fully address the critique. Return only the improved draft."
        )
    draft = chat(prompt)
    n = state['iterations'] + 1
    print(f"\n--- DRAFT {n} ---\n{draft}")
    return {'draft': draft, 'iterations': n}


def reflect(state):
    prompt = (
        f"Task: {state['task']}\n\n"
        f"Draft to evaluate:\n{state['draft']}\n\n"
        "You are a strict critic. If the draft fully and excellently satisfies the task, "
        "reply with exactly the word APPROVED and nothing else. "
        "Otherwise, give specific, actionable feedback on what to fix (do NOT rewrite it)."
    )
    critique = chat(prompt)
    print(f"\n--- CRITIQUE {state['iterations']} ---\n{critique}")
    return {'critique': critique}


## Task 2 — Wire the reflection loop
Build the graph: `START → generate → reflect`, then a **conditional edge** from `reflect` that ends if the critique is `APPROVED` **or** `iterations >= MAX_ITERS`, otherwise loops back to `generate`.
<br>Write a `should_continue(state)` that returns `END` or `'generate'`, and compile to `app`.
<br>Hint: `g.add_conditional_edges('reflect', should_continue, {'generate': 'generate', END: END})`. The `MAX_ITERS` cap is your loop guard (same instinct as P2-A6 `max_steps` / P4-A3 `recursion_limit`).

In [3]:
# TODO: should_continue(state) + build/compile the generate<->reflect graph as `app`

def should_continue(state):
    if state['critique'].strip().startswith('APPROVED'):   # critic is satisfied
        return END
    if state['iterations'] >= MAX_ITERS:                   # loop guard — don't revise forever
        return END
    return 'generate'                                      # otherwise, revise again

g = StateGraph(ReflectState)
g.add_node('generate', generate)
g.add_node('reflect', reflect)
g.add_edge(START, 'generate')
g.add_edge('generate', 'reflect')
g.add_conditional_edges('reflect', should_continue, {'generate': 'generate', END: END})

app = g.compile()
print('graph compiled')


graph compiled


## Task 3 — Run it and watch the draft improve
Invoke `app` with a task that benefits from revision, e.g. *"Explain what an API is to a complete non-technical beginner, in 3 sentences, using a real-world analogy and no jargon."* Start state: `{'task': ..., 'draft': '', 'critique': '', 'iterations': 0}`.
<br>Watch the printed drafts + critiques across iterations, then print the final `draft`.
<br>Goal: see the output measurably improve from draft 1 → final.

In [4]:
# TODO: run app on a revision-worthy task; observe drafts improving; print final draft

task = ("Explain what an API is to a complete non-technical beginner, in 3 sentences, "
        "using a real-world analogy and no jargon.")

result = app.invoke({'task': task, 'draft': '', 'critique': '', 'iterations': 0})

print("\n========== FINAL DRAFT ==========")
print(result['draft'])
print(f"\n(took {result['iterations']} iteration(s))")



--- DRAFT 1 ---
An API is like a waiter at a restaurant. You give the waiter your order (request), and they take it to the kitchen (the service or system), where your food is prepared. When your meal is ready, the waiter brings it back to you, allowing you to enjoy your food without needing to know what happens in the kitchen.

--- CRITIQUE 1 ---
APPROVED

========== FINAL DRAFT ==========
An API is like a waiter at a restaurant. You give the waiter your order (request), and they take it to the kitchen (the service or system), where your food is prepared. When your meal is ready, the waiter brings it back to you, allowing you to enjoy your food without needing to know what happens in the kitchen.

(took 1 iteration(s))


## Task 4 — Explain (in your own words)
1. Why can a model's *critique-then-revise* output beat its first attempt, even though it's the same model with no new information?
2. Reflection costs extra LLM calls per iteration. How would you decide *when* it's worth it, and how would you set the stopping condition well (think: the `APPROVED` signal, `MAX_ITERS`, and evals)?

> _your answer here_
1. Because generating and evaluating are different, asymmetric tasks — and the model is often better at the second. A first attempt is produced token-by-token in one forward pass with no chance to step back; flaws (jargon slipped in, a weak analogy, ignoring the "3 sentences" constraint) get baked in. When the model then critiques, it reads the finished draft as a fixed object and checks it against the task — a focused, narrower job that surfaces concrete problems. The revise step is no longer "write a good explanation from nothing," it's "fix these specific named issues," which is much easier. So even with no new external information, restructuring the work into attempt → spot defects → repair lets the same weights produce a better result — it's spending extra compute/attention on the hard parts rather than relying on one lucky pass.

2. Reflection multiplies cost (each iteration is a generate and a critique call, so ~2× per loop), so it's worth it only when quality matters more than latency/cost and the output genuinely benefits from revision — nuanced writing, code, anything with explicit constraints to satisfy. It's wasteful for simple, factual, or already-reliable outputs where draft 1 is fine. For the stopping condition I'd use two complementary guards (both present here): the APPROVED signal lets it stop early the moment the critic is satisfied, so I don't pay for needless extra loops; MAX_ITERS is the hard ceiling that guarantees termination and bounds cost even if the critic is never happy (the same loop-guard instinct as max_steps / recursion_limit). The risk is a miscalibrated critic — too lenient (approves junk) or too harsh (never approves, always burns all iterations) — so I'd tune and eval it: on a labelled set, measure whether APPROVED correlates with actually-good outputs (judge-graded, W2-A2 style) and track how much quality improves per iteration. If draft 1 → final shows little lift, reflection isn't earning its cost; if APPROVED fires on bad drafts, the critic prompt needs tightening.